In [22]:
# import packages and data

import pandas as pd
import os

path_data = 'C:/Users/skar/Box/saura_self/Proj - Water tool analysis/data/output'
fname = 'survey_effluent.xlsx'
sname = 'Sheet1'

path_out = 'C:/Users/skar/Box/saura_self/Proj - Water tool analysis/data/output'

df = pd.read_excel(os.path.join(path_data, fname), sheet_name=sname)

#print(df['PREDICTED_WRRF_TT_CODE'].unique())
[columns for columns in df.columns]

['CWNS_ID',
 'FACILITY_ID',
 'FACILITY_NAME',
 'LATITUDE',
 'LONGITUDE',
 'CITY',
 'STATE_CODE',
 'AUTHORITY_NAME',
 'COUNTY_NAME',
 'DESIGN_FLOW',
 'DESIGN_FLOW_UNITS',
 'ACTUAL_FLOW_MIN',
 'ACTUAL_FLOW_MAX',
 'ACTUAL_FLOW_AVG',
 'ACTUAL_FLOW_UNITS',
 'PREDICTED_WRRF_TT_CODE',
 'TT_IDENTIFIED',
 'Assigned TT',
 'BOD5_MIN',
 'BOD5_MAX',
 'BOD5_AVG',
 'BOD5_UNITS',
 'CBOD5_MIN',
 'CBOD5_MAX',
 'CBOD5_AVG',
 'CBOD5_UNITS',
 'SS_MIN',
 'SS_MAX',
 'SS_AVG',
 'SS_UNITS',
 'VSS_MIN',
 'VSS_MAX',
 'VSS_AVG',
 'VSS_UNITS',
 'TS_MIN',
 'TS_MAX',
 'TS_AVG',
 'TS_UNITS',
 'TSS_MIN',
 'TSS_MAX',
 'TSS_AVG',
 'TSS_UNITS',
 'VTS_MIN',
 'VTS_MAX',
 'VTS_AVG',
 'VTS_UNITS',
 'TKN_MIN',
 'TKN_MAX',
 'TKN_AVG',
 'TKN_UNITS',
 'TN_MIN',
 'TN_MAX',
 'TN_AVG',
 'TN_UNITS',
 'NH3_N_MIN',
 'NH3_N_MAX',
 'NH3_N_AVG',
 'NH3_N_UNITS',
 'NO3_N_MIN',
 'NO3_N_MAX',
 'NO3_N_AVG',
 'NO3_N_UNITS',
 'NO2_N_MIN',
 'NO2_N_MAX',
 'NO2_N_AVG',
 'NO2_N_UNITS',
 'NO3_N+NO2_N_MIN',
 'NO3_N+NO2_N_MAX',
 'NO3_N+NO2_N_AVG',
 'N

In [ ]:
# Flow rate method

path = "C:/Users/skar/repos/wwtp_energy_comparison/input_data/flow_method/"
df1 = df[['CWNS_ID', 'TT_IDENTIFIED', 'PREDICTED_WRRF_TT_CODE', 'ACTUAL_FLOW_AVG', 
          'Electricity_consumed_onsite_kWh_per_m3',
          'Electricity_purchased_from_utility_kWh_per_m3',
          'Electricity_produced_onsite_kWh_per_m3',
          ]].copy()

# Assign flow categories based on EPRI flow categories
"""
LESS THAN 2
2 TO 4
4 TO 7
7 TO 16
16 TO 46
46 TO 100
100 AND ABOVE
"""
def assign_flow_category(flow_rate):
    if flow_rate < 2:
        return 'LESS THAN 2'
    elif flow_rate >= 2 and flow_rate < 4:
        return '2 TO 4'
    elif flow_rate >= 4 and flow_rate < 7:
        return '4 TO 7'
    elif flow_rate >= 7 and flow_rate < 16:
        return '7 TO 16'
    elif flow_rate >= 16 and flow_rate < 46:
        return '16 TO 46'
    elif flow_rate >= 46 and flow_rate < 100:
        return '46 TO 100'
    else:
        return '100 AND ABOVE'
df1['FLOW_CAT_MGD'] = df1['ACTUAL_FLOW_AVG'].apply(assign_flow_category)

#import electricity intensity values based on flow rate from EPRI
el_flow = pd.read_excel(path + 'epri_flow_ei.xlsx', sheet_name = 'EPRI_FLOW_EI')

# Merge electricity intensity values with flow categories
df1 = df1.merge(el_flow, how = 'left', left_on = 'FLOW_CAT_MGD', right_on = 'Average Daily Flow (MGD)')

# Save output
df1.to_csv(os.path.join(path_out, 'Electricity_by_flow_method.csv'), index = False)

In [32]:
# Effluent Treatment level method (A)

# TTs with E, F, G are assigned 'Advanced Treatment'
# TTs with just A or B is assigned 'Primmary' and 'Advanced Primary' respectively
# TTs with sludge treatment or incineration are assigned 'Secondary'

dict_treatment_levels = {
    '*AG2' : 'Advanced Treatment', 
    '*B6, *B' : 'Secondary',  
    '*AEF2e' : 'Advanced Treatment',    
    '*AE5' : 'Advanced Treatment',    
    '*AF2' : 'Advanced Treatment',    
    '*CE5' : 'Advanced Treatment',    
    '*B1e' : 'Secondary',
    '*EF1' : 'Advanced Treatment',    
    '*A1e' : 'Secondary',    
    '*AF1' : 'Advanced Treatment',     
    '*A1' : 'Secondary',     
    '*AEF1' : 'Advanced Treatment',
    '*B' : 'Advanced Primary',    
    '*C1e' : 'Secondary',   
    '*AE6e' : 'Advanced Treatment',   
    '*EF1e' : 'Advanced Treatment',      
    '*A' : 'Primary', 
    '*ACEG1e' : 'Advanced Treatment',    
    '*A6e' : 'Secondary',   
    '*AE1e' : 'Advanced Treatment',
    '*AC1e' : 'Advanced Treatment',     
    '*A5' : 'Secondary',    
    '*BE5' : 'Advanced Treatment'
}

df2 = df[['CWNS_ID', 'TT_IDENTIFIED', 'PREDICTED_WRRF_TT_CODE', 'ACTUAL_FLOW_AVG', 
          'BOD5_AVG',
          'Electricity_consumed_onsite_kWh_per_m3',
          'Electricity_purchased_from_utility_kWh_per_m3',
          'Electricity_produced_onsite_kWh_per_m3',
          ]].copy()
# remove rows with missing PREDICTED_WRRF_TT_CODE
df2 = df2[df2['PREDICTED_WRRF_TT_CODE'].notna()]

def assign_treatment_level(tt_code):
    for key in dict_treatment_levels.keys():
        if key in tt_code:
            return dict_treatment_levels[key]
    return 'Unknown'
df2['Treatment_Level'] = df2['PREDICTED_WRRF_TT_CODE'].apply(assign_treatment_level)

# Check on Primary assignment with BOD concentraiton for each row of df2
mask_primary = df2['Treatment_Level'] == 'Primary'
mask_low_bod = df2['BOD5_AVG'] < 45

df2.loc[mask_primary & mask_low_bod, 'Treatment_Level'] = 'Primary (45mg/l< BOD)'
df2.loc[mask_primary & ~mask_low_bod, 'Treatment_Level'] = 'Advanced Primary'

    
# Read Electricity intensity data
path = 'C:\\Users\\skar\\repos\\wwtp_energy_comparison\\input_data\\effluent_methods\\'
el_treatment = pd.read_excel(path + 'epri_effluent_a_ei.xlsx', sheet_name = 'Sheet1')

# Merge electricity intensity values with treatment levels
df2 = df2.merge(el_treatment, how = 'left', left_on = 'Treatment_Level', right_on = 'Effluent Treatment Level')

# Drop Treatment_level column
df2 = df2.drop(columns = ['Treatment_Level'])

# Save output
df2.to_csv(os.path.join(path_out, 'Electricity_by_effluent_method_A.csv'), index = False)


In [ ]:
# Effluent Treatment level method (B)